# Bias-variance tradeoff

_Machine Learning_

**Too simple underfits. Too complex overfits. Aim for the middle.**

Every model has two ways to be wrong.

     Bias: the model is too simple and misses the real pattern. This is underfitting.

---

This notebook makes the tradeoff visible: we fit polynomials of growing degree and watch train error and test error pull apart. Run each cell, read the note above it, and look for the degree where test error bottoms out. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, and the target is a continuous value.

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target summary:")
print(data.target.describe())
data.frame.head()

## Reference implementation — scikit-learn

We fit polynomial models of increasing degree to a noisy sine curve and compare **train** error against **test** error. We build it in three steps: (1) make a controlled synthetic dataset, (2) split into train and test, (3) sweep the polynomial degree and watch the two errors diverge.

### Step 1 — Build a controlled noisy dataset

We generate data from a known truth — a sine curve — plus a little Gaussian noise. Because we *know* the true function is a smooth sine, we know in advance that a low-degree model will **underfit** (high bias) and a very high-degree one will **overfit** the noise (high variance).

Seeding the random generator keeps the dataset reproducible.

In [ ]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(0)             # seeded so results are reproducible

X = rng.uniform(-3, 3, size=(120, 1))      # 120 inputs spread over [-3, 3]
true_signal = np.sin(X[:, 0])              # the underlying truth we want to recover
noise = rng.normal(0, 0.3, size=120)       # measurement noise
y = true_signal + noise                     # observed targets = signal + noise

### Step 2 — Hold out a test set

We split off 30% of the data as a **test** set the model never trains on. The whole point of the exercise is the gap between training error (how well the model fits data it has seen) and test error (how well it generalises). Without a held-out set we could not see overfitting at all.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)

### Step 3 — Sweep the polynomial degree

For each degree we build a pipeline that expands the input into polynomial features and fits a linear model, then record train and test MSE. Watch the two columns: train error keeps falling as degree grows, but test error falls, bottoms out, then **rises** again once the model starts memorising noise. That U-shape in test error is the bias-variance tradeoff.

In [ ]:
for d in [1, 3, 9, 15]:
    model = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(Xtr, ytr)

    tr = mean_squared_error(ytr, model.predict(Xtr))   # error on data it trained on
    te = mean_squared_error(yte, model.predict(Xte))   # error on unseen data

    print("degree %2d  train MSE %.3f  test MSE %.3f" % (d, tr, te))

## Visualize the data & results

_Fitting diabetes progression from BMI with growing polynomial degree, where is the sweet spot between underfitting and overfitting?_

Now we repeat the sweep on real data and plot the result. We do it in two steps: (1) load one real feature and split it, (2) fit a finer grid of degrees and plot train vs test error so the U-shape is visible.

### Step 1 — Load one real feature and split

We use the diabetes dataset and keep a single feature — BMI (column 2) — so the relationship is easy to reason about. The target is disease progression. As before we hold out 30% as a test set so we can measure generalisation, not just fit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

db = load_diabetes()
x = db.data[:, 2:3]                # BMI (kept as a 2D column)
y = db.target                      # disease progression

Xtr, Xte, ytr, yte = train_test_split(x, y, test_size=0.3, random_state=0)

### Step 2 — Fit a grid of degrees and plot the two curves

We sweep a finer grid of degrees, recording train and test MSE for each. Adding `StandardScaler` to the pipeline keeps the high-degree polynomial features numerically well-behaved. The plot shows the signature picture: the train curve sinks steadily while the test curve dips and then climbs — the gap between them is variance, and the lowest point of the test curve is the sweet spot.

In [ ]:
degrees = [1, 2, 3, 5, 7, 9, 12, 15]
tr = []     # train MSE per degree
te = []     # test MSE per degree

for d in degrees:
    model = make_pipeline(PolynomialFeatures(d),
                          StandardScaler(),
                          LinearRegression()).fit(Xtr, ytr)
    tr.append(mean_squared_error(ytr, model.predict(Xtr)))
    te.append(mean_squared_error(yte, model.predict(Xte)))

plt.plot(degrees, tr, color="#4ea1ff", marker="o", label="train MSE")
plt.plot(degrees, te, color="#ff7b72", marker="o", label="test MSE")
plt.xlabel("polynomial degree")
plt.ylabel("mean squared error")
plt.title("Train vs test error")
plt.legend()
plt.show()